# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [3]:
!pip install pandas
import pandas as pd
import os

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
    --------------------------------------- 0.2/9.9 MB 6.1 MB/s eta 0:00:02
   --- ------------------------------------ 0.8/9.9 MB 12.7 MB/s eta 0:00:01
   ------ --------------------------------- 1.6/9.9 MB 17.2 MB/s eta 0:00:01
   ------------ --------------------------- 3.1/9.9 MB 22.2 MB/s eta 0:00:01
   -------------------- ------------------- 5.1/9.9 MB 25.2 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 28.0 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.9 MB 30.8 MB/s eta 0:00:01
   ---------------------------------------  9.9/9.9 MB 31.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 28.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.5/12.6 MB 47.6 MB/s eta 0:00:01
   --------- ------------------------------ 2.9/12.6 MB 45.9 MB/s eta 0:00:01
   ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.4 which is incompatible.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.4.4 which is incompatible.
scipy 1.11.4 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.4.4 which is incompatible.


## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, `date`. All else can be blank. `type` defaults to "Talk" 
- `date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. 
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talks/YYYY-MM-DD-[url_slug]`
    - The combination of `url_slug` and `date` must be unique, as it will be the basis for your filenames

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [4]:
!cat talks.tsv

'cat' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [18]:
talks = pd.read_csv("talks.tsv", sep="\t", header=0)
talks

,title,type,url_slug,venue,date,location,talk_url,description
0,NepoIP/MM: Toward accurate biomolecular simula...,Conference Talk,NepoIPMM,Georgia World Congress Center,3/23/2026,"Atlanta, Georgia",https://scimeetings.acs.org/exhibit/NepoIPMM-T...,ACS Spring 2026


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [19]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [20]:
loc_dict = {}

for row, item in talks.iterrows():
    
    md_filename = str(item.date) + "-" + str(item.url_slug) + ".md"
    html_filename = str(item.date) + "-" + str(item.url_slug) 
    year = item.date[:4]
    
    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    
    if len(str(item.type)) > 3:
        md += 'type: "' + item.type + '"\n'
    else:
        md += 'type: "Talk"\n'
    
    md += "permalink: /talks/" + html_filename + "\n"
    
    if len(str(item.venue)) > 3:
        md += 'venue: "' + item.venue + '"\n'
        
    if len(str(item.location)) > 3:
        md += "date: " + str(item.date) + "\n"
    
    if len(str(item.location)) > 3:
        md += 'location: "' + str(item.location) + '"\n'
           
    md += "---\n"
    
    
    if len(str(item.talk_url)) > 3:
        md += "\n[More information here](" + item.talk_url + ")\n" 
        
    
    if len(str(item.description)) > 3:
        md += "\n" + html_escape(item.description) + "\n"
        
        
    md_filename = os.path.basename(md_filename)
    #print(md)
    
    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [21]:
!ls ../_talks

'ls' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [22]:
!cat ../_talks/2013-03-01-tutorial-1.md

'cat' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���
